In [16]:
import pandas as pd
import numpy as np
np.set_printoptions(formatter={'float': lambda x: f"{x:.6g}"})


### 1. Функция Лагранжа

$$
L(w, \lambda) = w^T K w - \lambda \left( \sum_{i=1}^n w_i - 1 \right)
$$

### 2. Условия стационарности

$$
\frac{\partial L}{\partial w} = 2K w - \lambda \mathbf{1} = 0 \quad\Rightarrow\quad K w = \frac{\lambda}{2} \mathbf{1}
$$

$$
\frac{\partial L}{\partial \lambda} = \sum_{i=1}^n w_i - 1 = 0 \quad\Rightarrow\quad \mathbf{1}^T w = 1
$$

### 3. Выражение весов через обратную матрицу

$$
w = \mu K^{-1} \mathbf{1}, \qquad \mu = \frac{\lambda}{2}
$$

### 4. Нахождение множителя

$$
\mathbf{1}^T w = \mu (\mathbf{1}^T K^{-1} \mathbf{1}) = 1 \quad\Rightarrow\quad \mu = \frac{1}{\mathbf{1}^T K^{-1} \mathbf{1}}
$$

### 5. Вектор оптимальных весов

$$
w = \frac{K^{-1} \mathbf{1}}{\mathbf{1}^T K^{-1} \mathbf{1}}
$$

### 6. Дисперсия портфеля с минимальной дисперсией

$$
\sigma_p^2 = w^T K w = \frac{1}{\mathbf{1}^T K^{-1} \mathbf{1}}
$$

### P. S. Взятие производной по вектору

$$
\frac{\partial}{\partial w_k} \left( \sum_{i,j} w_i K_{ij} w_j \right)
= \sum_{i,j} K_{ij} \left( \frac{\partial w_i}{\partial w_k} w_j + w_i \frac{\partial w_j}{\partial w_k} \right).
$$

Так как $\frac{\partial w_i}{\partial w_k} = \delta_{ik}$, где $\delta_{ik}$ равно 1 при $i=k$ и 0 в противном случае, то:


* первое слагаемое:  
  $$
  \sum_{i,j} K_{ij} \delta_{ik} w_j = \sum_{j} K_{k j} w_j;
  $$

* второе слагаемое:  
  $$
  \sum_{i,j} K_{ij} w_i \delta_{jk} = \sum_{i} K_{i k} w_i.
  $$

Итого:

$$
\frac{\partial}{\partial w_k} (w^T K w)
= \sum_{j=1}^n K_{k j} w_j + \sum_{i=1}^n K_{i k} w_i.
$$

$$
\frac{\partial}{\partial w_k} \left( \sum_{i,j} w_i K_{ij} w_j \right)
= \sum_{i,j} K_{ij} \left( \frac{\partial w_i}{\partial w_k} w_j + w_i \frac{\partial w_j}{\partial w_k} \right) \nonumber \\
= \sum_{j} K_{k j} w_j + \sum_{i} K_{i k} w_i.
$$
При симметричности матрицы $K$ получаем:
$$
\frac{\partial}{\partial w_i} (w^T K w) = 2 \sum_{j=1}^n K_{i j} w_j.
$$

In [2]:
class Min_dispersion:
    def __init__(self, w: np.ndarray, K: np.ndarray):
        self.w = w
        self.K = K
        self.n = len(self.w)
        self.new_w = None
        self.K_inv = None
        self.dispersion = None

    def count_K_inv(self):          # подсчёт обратной матрицы
        self.K_inv = np.linalg.inv(self.K)
    
    def optimize_dispersion(self):  # подсчёт w, Q
        if self.K_inv is None:
            self.count_K_inv()
        ones = np.ones(self.n)      # вектор из n единиц
        numerator = self.K_inv @ ones
        denominator = ones @ self.K_inv @ ones
        self.new_w = numerator / denominator
        self.dispersion = 1.0 / denominator

                                    # получение w, Q 
    def get_new_vector(self) -> np.array:
        if self.new_w is None:
            raise ValueError("Сначала вызовите optimize_dispersion()")
        return self.new_w
    
    def get_new_dispersion(self) -> float:
        if self.dispersion is None:
            raise ValueError("Сначала вызовите optimize_dispersion()")
        return np.sqrt(self.dispersion) * 100   # в формате процентов


### Функция load_prices
 Получает на вход таблицу данных CSV (формат MOEX), преобразует в таблицу "дата - цена закрытия дня - тикер" и заполняет пропуски, в случае, когда цена закрытия не определена.
 Данные взяты за 1 год.
##### Требуется запустить один раз для загрузки данных в новую таблицу.

In [ ]:
def load_prices(csv_path: str, price_col: str = 'close', date_col: str = 'begin',
                ticker_col: str = 'ticker', fill_method: str = 'ffill',
                handle_duplicates: str = 'first') -> pd.DataFrame:
    # 1. Загрузка
    df = pd.read_csv(csv_path)

    # 2. Проверка наличия необходимых колонок
    required_cols = [date_col, ticker_col, price_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"В файле отсутствуют колонки: {missing}")

    # 3. Преобразование даты
    df[date_col] = pd.to_datetime(df[date_col])

    # 4. Сортировка для предсказуемости
    df = df.sort_values([date_col, ticker_col])

    # 5. Pivot с агрегацией на случай дубликатов
    pivot_df = df.pivot_table(index=date_col, columns=ticker_col,
                              values=price_col, aggfunc=handle_duplicates)

    # 6. Сортировка индекса по дате
    pivot_df = pivot_df.sort_index()

    # 7. Заполнение пропусков
    if fill_method == 'ffill':
        pivot_df = pivot_df.ffill()
    elif fill_method == 'bfill':
        pivot_df = pivot_df.bfill()
    elif fill_method == 'ffill+bfill':
        pivot_df = pivot_df.ffill().bfill()
    # иначе (NaN) – оставляем как есть

    # 8. Убираем строки, где все цены NaN (если такие остались после заполнения)
    pivot_df = pivot_df.dropna(how='all')

    return pivot_df


prices = load_prices('10years_data_1d_interval.csv', fill_method='ffill+bfill')
print("Размер таблицы цен:", prices.shape)
print(prices.head())
prices.to_csv('prices_processed.csv')


Размер таблицы цен: (253, 200)
ticker       ABIO  ABRD   AFKS   AFLT    AGRO    AKRN   ALRS  AMEZ   APTK  \
begin                                                                       
2016-04-08  21.05  88.0  17.15  79.00  1148.0  3555.0  71.89  3.68  16.36   
2016-04-11  21.80  86.5  17.89  82.20  1140.0  3555.0  71.70  3.75  18.33   
2016-04-12  21.20  86.5  18.40  79.32  1114.0  3530.0  71.75  3.71  16.80   
2016-04-13  21.10  85.5  18.75  78.55  1132.0  3550.0  71.20  3.77  16.60   
2016-04-14  22.75  88.0  18.70  78.63  1160.0  3675.0  71.60  3.79  16.46   

ticker      AQUA  ...    VTBR  WTCM  WTCMP  YAKG   YKEN  YKENP   YRSB  YRSBP  \
begin             ...                                                          
2016-04-08  34.0  ...  373.25   8.0   6.20  19.9  0.343  0.314  101.0   80.0   
2016-04-11  34.9  ...  384.70   8.0   6.20  19.9  0.382  0.340  101.0   80.0   
2016-04-12  34.3  ...  375.50   8.0   6.20  19.9  0.373  0.340  101.0   80.0   
2016-04-13  33.9  ...  384.00

### 1. Логарифмическая доходность
$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

### 2. Ковариационная матрица
$$
K_{ij} = \frac{1}{n-1} \sum_{k=1}^{n} (r_i^{(k)} - \bar{r}_i)(r_j^{(k)} - \bar{r}_j)
$$

In [22]:
prices = pd.read_csv('prices_processed.csv', index_col=0, parse_dates=True)
prices_20 = prices.iloc[:, :20]
returns = np.log(prices_20 / prices_20.shift(1)).dropna()    

K = np.cov(returns, rowvar=False)
K = K * len(prices_20)  # ковариационная матрица за n дней

init_w = np.zeros(len(prices_20.columns))
init_w[0] = 1

portfolio = Min_dispersion(init_w, K)
portfolio.optimize_dispersion()

opt_w = portfolio.get_new_vector()
opt_dispersion = portfolio.get_new_dispersion()

print(prices_20)
print("min dispersion(%): ", opt_dispersion)
print("opt vector: ", opt_w)
print("sum of coords :", sum(opt_w)) 
# решить проблему с отрицательными координатами

             ABIO   ABRD   AFKS    AFLT    AGRO    AKRN   ALRS  AMEZ   APTK  \
begin                                                                         
2016-04-08  21.05   88.0  17.15   79.00  1148.0  3555.0  71.89  3.68  16.36   
2016-04-11  21.80   86.5  17.89   82.20  1140.0  3555.0  71.70  3.75  18.33   
2016-04-12  21.20   86.5  18.40   79.32  1114.0  3530.0  71.75  3.71  16.80   
2016-04-13  21.10   85.5  18.75   78.55  1132.0  3550.0  71.20  3.77  16.60   
2016-04-14  22.75   88.0  18.70   78.63  1160.0  3675.0  71.60  3.79  16.46   
...           ...    ...    ...     ...     ...     ...    ...   ...    ...   
2017-04-03  14.70  135.0  23.00  168.30   703.0  3305.0  91.90  4.97   8.42   
2017-04-04  14.05  145.0  22.90  168.70   700.0  3286.0  92.00  5.08   8.38   
2017-04-05  14.35  149.5  22.75  174.95   707.0  3283.0  92.14  5.12   8.35   
2017-04-06  14.75  150.0  22.82  177.50   710.0  3310.0  91.55  5.10   8.34   
2017-04-07  14.20  150.0  22.35  171.90   713.0  333

## Critical-line method
### Создание угловых портфелей, описывающих границу эффективного множества.
Угловой портфель (corner portfolio) — это эффективный портфель, обладающий следующим свойством:
любая комбинация двух смежных угловых портфелей представляет собой третий портфель, лежащий на эффективной границе (в эффективном множестве) между этими двумя угловыми портфелями.

In [28]:
prices = pd.read_csv('prices_processed.csv', index_col=0, parse_dates=True)
n = 5
prices_n = prices.iloc[:, :n]
returns = np.log(prices_n / prices_n.shift(1)).dropna()    

K = np.cov(returns, rowvar=False)
K = K * 252  # ковариационная матрица n активов за год
K_inv = np.linalg.inv(K)


# средняя дневная доходность каждого актива
profit_day = returns.mean()

#средняя годовая доходность каждого актива
profit_year = profit_day * 252
pv = profit_year.values

I = [np.argmax(pv)]          # начальное активное множество (индекс актива с максимальной ожидаемой доходностью)
r_cur = pv[I[0]]             # текущая доходность
corners = []                 # список для хранения угловых портфелей (веса, доходность, риск)

# первый угловой портфель
w_full = np.zeros(n)
w_full[I[0]] = 1
corners.append((w_full, r_cur * 100, np.sqrt(K[I[0], I[0]]) * 100))

while True:
    if(len(I) == 1):
        i = I[0]
        sigma_ii = K[i, i]
        r_in_candidates = []
        for j in range(n):
            if j == i:
                continue
            sigma_jj = K[j, j]
            sigma_ij = K[i, j]
            denom = sigma_ii + sigma_jj - 2 * sigma_ij
            if denom <= 0:
                continue
            wj = (sigma_ii - sigma_ij) / denom
            r_in = pv[i] + wj * (pv[j] - pv[i])
            if r_in < pv[i]:
                r_in_candidates.append((r_in, j, wj))
        if r_in_candidates:
            # Выбираем кандидата с максимальным r_in
            r_next, j_best, wj_best = max(r_in_candidates, key=lambda x: x[0])
            # Веса второго углового портфеля
            w_full = np.zeros(n)
            w_full[i] = 1 - wj_best
            w_full[j_best] = wj_best
            # Добавляем второй угловой портфель в список
            corners.append((w_full.copy(), r_next * 100, np.sqrt(w_full @ K @ w_full) * 100))
            # Обновляем активное множество и текущую доходность
            I = [i, j_best]
            r_cur = r_next
        else:
            print("not new in_candidates")
            break
        continue

    # ковариационная подматрица, подвектор доходностей, единичный подвектор для множества I
    K_I = K[np.ix_(I, I)]
    pv_I = pv[I]
    ones_I = np.ones(len(I))

    # обратная ковариационная подматрица
    K_I_inv = np.linalg.inv(K_I)

    # рассчёт векторов и скаляров для решения системы уравнений
    a = K_I_inv @ pv_I
    b = K_I_inv @ ones_I

    A = pv_I @ a
    B = pv_I @ b
    C = ones_I @ b
    det = A * C - B * B
    
    if np.isclose(det, 0):
        print("error with det = 0")
        break

    k_r = (C * a - B * b) / det
    l_r = (A * b - B * a) / det
    
    # поиск кандидатов на выход из множества I
    r_out_candidates = []
    for i in range(len(I)):
        if np.isclose(k_r[i], 0):
            continue
        r_i = -l_r[i] / k_r[i]
        if r_i < r_cur:
            r_out_candidates.append((r_i, i))
    r_out_max = max(r_out_candidates, default=(-np.inf, None))[0]

    # поиск кандидатов на вход в множество I
    r_in_candidates = []
    outside = [j for j in range(n) if j not in I] # Все индексы вне I
    for j in outside:
        # вычисление коэффицентов для h_r
        Sigma_k_j = K[j, I] @ k_r
        Sigma_l_j = K[j, I] @ l_r
        alpha = Sigma_k_j - (C / det) * pv[j] + (B / det)
        beta = Sigma_l_j + (B / det) * pv[j] - (A / det)
        if np.isclose(alpha, 0):
            continue
        r_j = -beta / alpha
        if r_j < r_cur:
            r_in_candidates.append((r_j, j))
    r_in_max = max(r_in_candidates, default=(-np.inf, None))[0]
    
    # определение r_next
    r_next = max(r_out_max, r_in_max)
    if r_next == -np.inf:
        print("dispersion is min") # достигнут портфель с минимальной дисперсией
        break
    
    # вычисление веса при r_next для текущего I
    w_I = k_r * r_next + l_r
    
    # достроение подвектора весов до размерности n
    w_full = np.zeros(n)
    w_full[I] = w_I
    
    # сохранение углового портфеля
    corners.append((w_full.copy(), r_next * 100, np.sqrt(w_full @ K @ w_full) * 100))
    
    # обновление I
    new_I = []
    # удаление активов, которые обнулились
    for idx, i in enumerate(I):
        if not np.isclose(w_I[idx], 0):
            new_I.append(i)
    # добавление активов, которые вошли
    for r_j, j in r_in_candidates:
        if np.isclose(r_j, r_next):
            new_I.append(j)
    # удаление возможных дубликатов (при удалении и добавлении одного и того же актива)
    I = list(set(new_I))
    
    r_cur = r_next
    
    # завершение, если множество I пусто (но метод не должен приводить к пустому I, только из-за численных погрешностей)
    if len(I) == 0:
        print("I is empty")
        break

print("Список эффективных портфелей:")
for i, (weights, prof, risk) in enumerate(corners, 1):
    weights_str = np.array2string(weights, formatter={'float': lambda x: f"{x:.6g}"})
    print(f"Портфель {i}:")
    print(f"  Веса:     {weights_str}")
    print(f"  Доходность: {prof:.6g}%")
    print(f"  Риск:      {risk:.6g}%")
    print()

dispersion is min
Список эффективных портфелей:
Портфель 1:
  Веса:     [0 0 0 1 0]
  Доходность: 77.7465%
  Риск:      31.5189%

Портфель 2:
  Веса:     [0 0.448973 0 0.551027 0]
  Доходность: 66.7841%
  Риск:      23.4844%

Портфель 3:
  Веса:     [0 1 0 0 0]
  Доходность: 53.3298%
  Риск:      34.8881%

Портфель 4:
  Веса:     [0 0.399134 0.600866 0 0]
  Доходность: 37.1984%
  Риск:      21.9984%

Портфель 5:
  Веса:     [0 0.268829 0.731171 0 0]
  Доходность: 33.7001%
  Риск:      22.7687%

Портфель 6:
  Веса:     [0 0.268829 0.731171 0 0]
  Доходность: 33.7001%
  Риск:      22.7687%

Портфель 7:
  Веса:     [0 0 1 0 0]
  Доходность: 26.4828%
  Риск:      28.416%

Портфель 8:
  Веса:     [0.359382 0 0.640618 0 0]
  Доходность: 2.818%
  Риск:      23.3719%

Портфель 9:
  Веса:     [0.54447 0 0.45553 0 0]
  Доходность: -9.36984%
  Риск:      24.8099%

Портфель 10:
  Веса:     [0.54447 0 0.45553 0 0]
  Доходность: -9.36984%
  Риск:      24.8099%

Портфель 11:
  Веса:     [0.717279 0 0

### Нахождение k-го углового портфеля

#### 1. Активное множество и параметризация

Пусть активное множество $I$ состоит из $m$ активов. Для фиксированного $I$ решаем задачу:

$$
\min_{w} \frac12 w^T \Sigma w \quad \text{при} \quad
\sum_{i \in I} w_i = 1,\quad
\sum_{i \in I} w_i \mu_i = r,
$$

где $r$ — заданная доходность. Остальные веса равны нулю.

#### 2. Функция Лагранжа

$$
L = \frac12 w^T \Sigma w - \lambda \left( \sum_{i \in I} w_i \mu_i - r \right) - \gamma \left( \sum_{i \in I} w_i - 1 \right).
$$

Здесь $\lambda$ и $\gamma$ — множители Лагранжа.

#### 3. Условия стационарности

$$
(\Sigma w)_i - \lambda \mu_i - \gamma = 0, \quad \forall i \in I.
$$

В матричной форме:

$$
\Sigma_I w_I = \lambda \mu_I + \gamma \mathbf{1}_I,
$$

где $\Sigma_I$ — ковариационная подматрица, $\mu_I$ — вектор доходностей, $\mathbf{1}_I$ — вектор единиц.

#### 4. Решение относительно $w_I$

$$
w_I = \lambda a + \gamma b,
$$

где

$$
a = \Sigma_I^{-1} \mu_I, \quad b = \Sigma_I^{-1} \mathbf{1}_I.
$$

#### 5. Множители через $r$

Подставляем $w_I$ в ограничения:

$$
\mu_I^T w_I = r, \quad \mathbf{1}_I^T w_I = 1.
$$

Получаем систему:

$$
\begin{cases}
A \lambda + B \gamma = r,\\
B \lambda + C \gamma = 1,
\end{cases}
$$

где

$$
A = \mu_I^T \Sigma_I^{-1} \mu_I, \quad
B = \mu_I^T \Sigma_I^{-1} \mathbf{1}_I, \quad
C = \mathbf{1}_I^T \Sigma_I^{-1} \mathbf{1}_I.
$$

Определитель системы:

$$
\Delta = AC - B^2 > 0.
$$

Решение:

$$
\lambda = \frac{r C - B}{\Delta}, \quad
\gamma = \frac{A - r B}{\Delta}.
$$

#### 6. Веса как линейная функция от $r$

$$
w_I(r) = k_r r \;+\; l_r, \quad
k_r =  \left( \frac{C a - B b}{\Delta} \right), \quad
l_r = \left( \frac{A b - B a}{\Delta} \right).
$$

Таким образом, $w_I(r)$ линейно зависит от $r$ на интервале постоянства $I$.

#### 7. Поиск критической доходности $r_k$

Двигаемся вниз от $r_{k-1} = \max \mu$ до $r < r_{k-1}$. В какой-то момент:

1. **Выход актива из $I$**  
   Для $i \in I$ решаем $w_i(r) = 0$, получаем $r_i^{out}$. Берём максимальное:  
   $$
   r_{out} = \max_{i \in I} r_i^{out}.
   $$

2. **Вход нового актива**  
   Для $j \notin I$ условие оптимальности:  
   $$
   (\Sigma w)_j - \lambda \mu_j - \gamma \ge 0.
   $$
   При движении вниз это выражение может стать нулём. Обозначим его $h_j(r)$:
   $$
   h_j(r) = (\Sigma_{k_j} r + \Sigma_{l_j}) - \lambda \mu_j - \gamma
   = \left( \Sigma_{k_j} - \frac{C}{\Delta} \mu_j + \frac{B}{\Delta} \right) r + \left( \Sigma_{l_j} + \frac{B}{\Delta} \mu_j - \frac{A}{\Delta}   \right).
   $$
   Решаем $h_j(r) = 0$, получаем $r_j^{in}$. Берём максимальное:  
   $$
   r_{in} = \max_{j \notin I} r_j^{in}.
   $$

**k-ый угловой портфель** соответствует:

$$
r_k = \max(r_{out}, r_{in}), \quad \text{при условии } r_k < r_{k-1}.
$$

Если событий несколько — обновляем $I$ (удаляем обнулившиеся, добавляем вошедшие).

#### 8. Вычисление $C_k$

Подставляем $r_k$ в $w_I(r)$. Веса вне $I$ равны нулю. Получаем угловой портфель:

$$
C_k = w_I(r_k).
$$

Его доходность: $r_k$.

#### 9. Дальнейший процесс

Повторяем для нового активного множества $I_k$, пока не дойдём до портфеля с минимальной дисперсией.

#### P.S. Откуда берётся условие оптимальности

Мы решаем задачу минимизации дисперсии портфеля с ограничениями на неотрицательность весов:

$$
\min_{w} \frac12 w^T \Sigma w \quad \text{при} \quad
\sum_{i=1}^n w_i = 1,\quad
\sum_{i=1}^n w_i \mu_i = r,\quad
w_i \ge 0.
$$

Функция Лагранжа (с учётом неравенств $w_j \ge 0$) записывается как:

$$
L = \frac12 w^T \Sigma w - \lambda \left( \sum_i w_i \mu_i - r \right) - \gamma \left( \sum_i w_i - 1 \right) - \sum_j \delta_j w_j,
$$

где $\delta_j \ge 0$ — множители Лагранжа для ограничений $w_j \ge 0$.

Условия стационарности по $w_j$ имеют вид:

$$
\frac{\partial L}{\partial w_j} = (\Sigma w)_j - \lambda \mu_j - \gamma - \delta_j = 0.
$$

Отсюда получаем:

$$
(\Sigma w)_j - \lambda \mu_j - \gamma = \delta_j \ge 0.
$$

Таким образом, для любого актива $j$ (входящего или не входящего в портфель) выражение $(\Sigma w)_j - \lambda \mu_j - \gamma$ должно быть неотрицательным. Если актив входит ($w_j > 0$), то $\delta_j = 0$ и равенство выполняется. Если актив не входит ($w_j = 0$), то выражение может быть строго положительным.

Зависимость от $r$ входит через $w(r)$, $\lambda(r)$ и $\gamma(r)$, которые линейны по $r$ на интервале постоянства активного множества.